In [2]:
%pip install fake-useragent

import os
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from fake_useragent import UserAgent

# Data directory configuration
DATA_DIR = "data"
RAW_DATA_DIR = os.path.join("..", DATA_DIR, "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Constants
BASE_URL = "https://www.transfermarkt.com"
HEADERS = {'User-Agent': UserAgent().random}
DELAY = 2  # seconds between requests to avoid rate limiting

# League IDs from Transfermarkt (you may need to update these)
LEAGUE_IDS = {
    'premier_league': 'GB1',
    'la_liga': 'ES1',
    'bundesliga': 'L1',
    'serie_a': 'IT1',
    'ligue_1': 'FR1',
    'brasileiro': 'BRA1',
    'eredivisie': 'NL1',
    'super_lig': 'TR1',
    'primeira_liga': 'PO1'
}

def get_league_players(league_id):
    """Get all players from a specific league"""
    url = f"{BASE_URL}/{league_id}/kader/wettbewerb/{league_id}/plus/1"
    
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        players = []
        # Find the table containing player data
        table = soup.find('table', {'class': 'items'})
        if not table:
            return players
            
        rows = table.find_all('tr', {'class': ['odd', 'even']})
        
        for row in rows:
            try:
                # Player name
                name_tag = row.find('td', {'class': 'hauptlink'}).find('a')
                name = name_tag.text.strip()
                
                # Market value
                value_tag = row.find('td', {'class': 'rechts hauptlink'})
                value = value_tag.text.strip() if value_tag else 'N/A'
                
                # Contract expiry
                contract_tag = row.find_all('td', {'class': 'zentriert'})[-1]
                contract = contract_tag.text.strip() if contract_tag else 'N/A'
                
                players.append({
                    'Player': name,
                    'MarketValue': value,
                    'ContractExpiry': contract
                })
            except Exception as e:
                print(f"Error processing player row: {e}")
                continue
                
        return players
        
    except Exception as e:
        print(f"Error scraping league {league_id}: {e}")
        return []

def scrape_all_leagues():
    """Scrape all specified leagues and combine results"""
    all_players = []
    
    for league_name, league_id in LEAGUE_IDS.items():
        print(f"Scraping {league_name}...")
        players = get_league_players(league_id)
        for player in players:
            player['League'] = league_name
        all_players.extend(players)
        time.sleep(DELAY)  # Be polite with delays
    
    # Create DataFrame
    df = pd.DataFrame(all_players)
    
    # Save to CSV in the specified directory
    output_path = os.path.join(RAW_DATA_DIR, 'transfermarkt_players.csv')
    df.to_csv(output_path, index=False)
    print(f"Data saved to {output_path}")
    
    return df

if __name__ == "__main__":
    df = scrape_all_leagues()
    print(f"Scraped {len(df)} players")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Scraping premier_league...
Error scraping league GB1: 404 Client Error: Not Found for url: https://www.transfermarkt.com/GB1/kader/wettbewerb/GB1/plus/1
Scraping la_liga...
Error scraping league ES1: 404 Client Error: Not Found for url: https://www.transfermarkt.com/ES1/kader/wettbewerb/ES1/plus/1
Scraping bundesliga...
Error scraping league L1: 404 Client Error: Not Found for url: https://www.transfermarkt.com/L1/kader/wettbewerb/L1/plus/1
Scraping serie_a...
Error scraping league IT1: 404 Client Error: Not Found for url: https://www.transfermarkt.com/IT1/kader/wettbewerb/IT1/plus/1
Scraping ligue_1...
Error scraping league FR1: 404 Client Error: Not Found for url: https://www.transfermarkt.com/FR1/kader/wettbewerb/FR1/plus/1
Scraping brasileiro...
Error scraping league BRA1: 404 Client Error: Not Found for url: https://www.transfermarkt.com/B